# 05 找出感染源 — 筆記本

第 04 章說明了*何時*、*何地*和*何人*。現在 Maya 要檢驗**為什麼**：颱風 Amihan 之後，哪些暴露讓鉤端螺旋體病更容易發生？Doc Mon 的團隊在河岸八個 barangay（描籠涯，菲律賓最基層的行政區，類似台灣的村里）訪問了 **120 位病例和 240 位對照**（每個病例，配上同一個 barangay 的兩位鄰居）。這項研究、這些人和每個數字都是合成的。

你會用 `pd.crosstab` 建立 2×2 表、手動計算勝算比和 95% 信賴區間、用 scipy 和 statsmodels 檢查、抓出被翻轉的表格，並用 Mantel–Haenszel 方法釐清一個干擾因子。

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

## 步驟 1 · 載入研究資料並檢視

為什麼用**病例對照研究**？即使在洪水之後，鉤端螺旋體病也不算常見，所以追蹤整座城市（一個*世代*）會需要訪問好幾萬人，才找得到幾百個病例。因此我們從已有的**病例**出發，從同樣的 barangay 挑選沒有生病的**對照**，並比較兩組在颱風之後做了什麼。

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm
from scipy.stats.contingency import odds_ratio
# BEGIN-CJK (中文字型設定：避免標籤顯示為方框)
import pathlib
import matplotlib.font_manager as fm
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass
_discovered = []
for _entry in fm.fontManager.ttflist:
    if any(_kw in _entry.name.lower() for _kw in ("cjk", "wenquanyi", "wqy")):
        if _entry.name not in _discovered:
            _discovered.append(_entry.name)
_preferred = ["Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP", "Noto Sans TC",
              "Microsoft JhengHei", "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
              "Heiti TC", "DejaVu Sans"]
_cjk_fonts = _discovered + [n for n in _preferred if n not in _discovered]
plt.rcParams["font.sans-serif"] = _cjk_fonts + [
    f for f in plt.rcParams.get("font.sans-serif", []) if f not in _cjk_fonts
]
plt.rcParams["axes.unicode_minus"] = False
del _discovered, _preferred, _cjk_fonts
# END-CJK

cc = pd.read_csv("data/synthetic/ph/lepto_case_control.csv")
print(len(cc), "位參與者：", cc["status"].value_counts().to_dict())
print("配對組數：", cc["matched_set"].nunique(), "| barangay：", ", ".join(sorted(cc["barangay"].unique())))
cc.head(6)

## 步驟 2 · Tala 搞錯了：上下顛倒的表格

> **問問 Tala：**「在 `cc` 裡，`waded_floodwater` 是 0/1，`status` 是 `case`/`control`。請做出 2×2 表，並計算涉水的勝算比。」

Tala 建立了這張表，並**依位置**讀取四個儲存格：左上是 *a*，右上是 *b*，依此類推。

In [ ]:
# Tala 的第一個答案：依「位置」讀取儲存格，沒有看標籤
t = pd.crosstab(cc["waded_floodwater"], cc["status"])
a, b = t.iloc[0, 0], t.iloc[0, 1]
c, d = t.iloc[1, 0], t.iloc[1, 1]
print(t)
print("Tala 的 OR =", round(a * d / (b * c), 2))

OR 0.18 的意思會是涉水**保護**了人。Aling Nena 會笑出來。在不知道答案的情況下，我們要怎麼抓到這個錯誤？

1. **讀標籤。** `crosstab` 把列依 0 在 1 之前排序，所以*第一*列是「**沒有**涉水」。Tala 的 *a* 是未暴露的病例，不是暴露的病例。公式假設的是一種順序；表格是另一種順序。
2. **檢查方向是否合理。** 在套用任何公式之前，先比較單純的百分比：涉水的病例比對照多嗎？如果是，OR 必定大於 1。

（把列*和*欄一起對調，也就是完整轉置，不會改變 OR。只翻轉其中一個，則會給出**倒數**：0.18 = 1 / 5.57。）

In [ ]:
waded_pct = pd.crosstab(cc["status"], cc["waded_floodwater"], normalize="index") * 100
print("涉過水的百分比：")
print(waded_pct[1].round(1))

四分之三的病例涉過水，對照則大約三分之一：OR 必定遠大於 1。現在來看安全的做法。我們用 `.loc[[1, 0], [1, 0]]` 把列和欄放進**固定且有標籤的順序**，並替它們命名，讓任何人都讀得懂這張表：

|  | 病例 | 對照 |
|---|---|---|
| **暴露** | a | b |
| **未暴露** | c | d |

$$OR = \frac{a \times d}{b \times c} \qquad
95\%\ CI = \exp\left(\ln OR \pm 1.96\sqrt{\tfrac1a+\tfrac1b+\tfrac1c+\tfrac1d}\right)$$

這個信賴區間是 *Woolf*（對數）方法。

In [ ]:
def two_by_two(data: pd.DataFrame, exposure: str) -> pd.DataFrame:
    """2x2 表：列為暴露／未暴露，欄為病例／對照。"""
    table = pd.crosstab(data[exposure], data["case"]).loc[[1, 0], [1, 0]]
    table.index = ["暴露", "未暴露"]
    table.columns = ["病例", "對照"]
    return table


table = two_by_two(cc, "waded_floodwater")
(a, b), (c, d) = table.to_numpy()
or_wading = a * d / (b * c)
se = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
low, high = math.exp(math.log(or_wading) - 1.96 * se), math.exp(math.log(or_wading) + 1.96 * se)

print(table, "\n")
print(f"a={a} b={b} c={c} d={d}")
print(f"OR = {or_wading:.2f}（95% CI {low:.2f}–{high:.2f}）")

用兩個獨立的函式庫來**檢查**。scipy 的 `odds_ratio` 必須指定 `kind="sample"`（它的預設是另一種估計式，*條件式* OR，會給出略有不同的數字）。statsmodels 的 `Table2x2` 使用同樣的 Woolf 區間。

In [ ]:
scipy_result = odds_ratio(table.to_numpy(), kind="sample")
scipy_ci = scipy_result.confidence_interval(confidence_level=0.95)
sm_table = sm.stats.Table2x2(table.to_numpy())
sm_low, sm_high = sm_table.oddsratio_confint()

print(f"手算        ：{or_wading:.2f}（{low:.2f}–{high:.2f}）")
print(f"scipy       ：{scipy_result.statistic:.2f}（{scipy_ci.low:.2f}–{scipy_ci.high:.2f}）")
print(f"statsmodels ：{sm_table.oddsratio:.2f}（{sm_low:.2f}–{sm_high:.2f}）")
print(f"scipy 預設值（條件式）：{odds_ratio(table.to_numpy()).statistic:.2f}")

assert math.isclose(scipy_result.statistic, or_wading) and math.isclose(sm_table.oddsratio, or_wading)
assert math.isclose(scipy_ci.low, low, rel_tol=1e-3) and math.isclose(sm_high, high, rel_tol=1e-3)

## 步驟 3 · 風險比還是勝算比？

**風險比**（risk ratio, RR）比較暴露組與未暴露組的疾病*風險*：$\frac{a/(a+b)}{c/(c+d)}$。這需要所有的風險族群，就像在世代研究裡那樣。在病例對照研究裡，要取多少對照是**我們自己選的**，所以我們這張表裡的「風險」是由我們的設計編出來的。看看如果每個病例配**一位**對照而不是兩位，會發生什麼事：「RR」會變動，OR 幾乎不動。

In [ ]:
def fake_rr(t: pd.DataFrame) -> float:
    (a, b), (c, d) = t.to_numpy()
    return (a / (a + b)) / (c / (c + d))


one_control = cc[(cc["case"] == 1) | ~cc.duplicated(["matched_set", "case"])]  # 只留第一位對照
for label, data in [("每個病例 2 位對照", cc), ("每個病例 1 位對照", one_control)]:
    t = two_by_two(data, "waded_floodwater")
    print(f"{label}：n = {len(data)} | OR = {sm.stats.Table2x2(t.to_numpy()).oddsratio:.2f}"
          f" | 'RR' = {fake_rr(t):.2f}  <- 不是真正的風險比")

所以：**世代研究 → RR；病例對照研究 → OR。** 當疾病在族群中罕見時，OR 會接近 RR，這就是為什麼我們仍能謹慎地說「大約五倍的風險」。

## 步驟 4 · 涉水者的雨鞋，以及 doxycycline

雨鞋只對**曾經**在水裡的人才重要，所以雨鞋這個問題只問涉水者（其他人的 `wore_boots` 是空白）。我們先限定在涉水者。Doxycycline 預防性投藥是在疏散中心及發給清理志工。OR **小於 1** 表示暴露在病例中較不常見：保護因子。

In [ ]:
waders = cc[cc["waded_floodwater"] == 1]
for label, data, exposure in [("穿雨鞋（僅限涉水者）", waders, "wore_boots"),
                              ("Doxycycline 預防性投藥", cc, "doxy_prophylaxis")]:
    t = two_by_two(data, exposure)
    result = sm.stats.Table2x2(t.to_numpy())
    lo, hi = result.oddsratio_confint()
    print(f"{label}：n = {len(data)}")
    print(t)
    print(f"OR = {result.oddsratio:.2f}（95% CI {lo:.2f}–{hi:.2f}）\n")

## 步驟 5 · 把所有暴露放在一張圖裡

> **問問 Tala：**「請用我的函式 `two_by_two(data, exposure)`（列為暴露／未暴露，欄為病例／對照），為幾個 0/1 欄位建立一張包含 a、b、c、d、OR 和 95% CI 的表，然後畫一張森林圖，x 軸用對數尺度，並在 OR = 1 畫一條線。」

`rats_near_home` 是一個有用的*虛無*暴露：老鼠對鉤端螺旋體病一般來說很重要，但在這項研究裡，在房子附近看到老鼠，並沒有區分出病例和對照。

In [ ]:
cc["male"] = (cc["sex"] == "M").astype(int)
EXPOSURES = [("waded_floodwater", cc), ("wore_boots", waders), ("doxy_prophylaxis", cc),
             ("skin_wound", cc), ("cleanup_work", cc), ("male", cc), ("rats_near_home", cc)]

rows = []
for exposure, data in EXPOSURES:
    t = two_by_two(data, exposure)
    result = sm.stats.Table2x2(t.to_numpy())
    lo, hi = result.oddsratio_confint()
    (a, b), (c, d) = t.to_numpy()
    label = exposure + ("（僅限涉水者）" if data is waders else "")
    rows.append({"暴露": label, "a": a, "b": b, "c": c, "d": d,
                 "OR": round(result.oddsratio, 2), "CI 下限": round(lo, 2), "CI 上限": round(hi, 2)})
summary = pd.DataFrame(rows).set_index("暴露")

fig, ax = plt.subplots(figsize=(7, 4))
y = list(range(len(summary)))[::-1]  # 第一個暴露放在最上面
for yi, (name, row) in zip(y, summary.iterrows()):
    # 紅色 = 危險因子，綠色 = 保護因子，灰色 = CI 跨過 1
    colour = "#CE1126" if row["CI 下限"] > 1 else "#2F7D32" if row["CI 上限"] < 1 else "#6E7882"
    ax.plot([row["CI 下限"], row["CI 上限"]], [yi, yi], color=colour, linewidth=2.5)
    ax.plot(row["OR"], yi, "o", color=colour, markersize=8)
ax.axvline(1, color="black", linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_xticks([0.2, 0.5, 1, 2, 5, 10], labels=["0.2", "0.5", "1", "2", "5", "10"])
ax.set_yticks(y, labels=summary.index)
ax.set_xlabel("勝算比（95% CI，對數尺度）")
ax.set_title("各暴露與鉤端螺旋體病，粗 OR（合成研究）")
plt.tight_layout()
plt.show()
summary

## 步驟 6 · 干擾作用：是涉水，還是清理？

Doc Mon 看著這張表問：*「清理志工涉水涉得比較多，不是嗎？而且清理工作本身也有風險：傷口、在泥巴裡待很久。涉水 OR 的一部分，會不會其實屬於清理工作？」*

**干擾因子**（confounder）是和暴露有關、*而且*本身就是該疾病的危險因子的第三個因素，所以它會把自己的效應混進粗 OR 裡。我們在**對照組**中檢查第一個關聯（它們讓我們看到各項暴露在族群中是怎麼一起出現的）。

In [ ]:
controls = cc[cc["case"] == 0]
print("依是否做清理工作，對照組中涉過水的百分比（%）：")
print((pd.crosstab(controls["cleanup_work"], controls["waded_floodwater"], normalize="index")[1] * 100).round(1))
print()
print("清理工作本身的粗 OR：", summary.loc["cleanup_work", "OR"])

做清理工作的人涉水的頻率高得多，而清理工作本身也與疾病有關。所以我們做**分層**：分別在做過清理工作的人和沒做過的人當中，計算涉水的 OR。然後我們用 **Mantel–Haenszel**（MH）勝算比把兩層合併，這是各層 OR 的加權平均：

$$OR_{MH} = \frac{\sum_i a_i d_i / n_i}{\sum_i b_i c_i / n_i}$$

其中 *i* 是分層，*n_i* 是該層的人數。

In [ ]:
numerator = denominator = 0.0
strata = []
for work, group in cc.groupby("cleanup_work"):
    t = two_by_two(group, "waded_floodwater")
    (a, b), (c, d) = t.to_numpy()
    n = a + b + c + d
    numerator += a * d / n
    denominator += b * c / n
    strata.append(t.to_numpy())
    label = "有做清理工作" if work == 1 else "沒做清理工作"
    print(f"{label:<18} a={a:>2} b={b:>2} c={c:>2} d={d:>3}  該層 OR = {a * d / (b * c):.2f}")

or_mh = numerator / denominator
print(f"\n粗 OR = {or_wading:.2f}   Mantel-Haenszel OR = {or_mh:.2f}")

用 statsmodels 的 `StratifiedTable` 來**檢查**，它也會給出 MH 信賴區間，以及一個檢定：各層 OR 之間的差異，是否超出機遇所能解釋的範圍。

In [ ]:
st = sm.stats.StratifiedTable(strata)
mh_low, mh_high = st.oddsratio_pooled_confint()
print(f"statsmodels MH OR = {st.oddsratio_pooled:.2f}（95% CI {mh_low:.2f}–{mh_high:.2f}）")
print(f"各層 OR 相等的檢定（Breslow-Day）：p = {st.test_equal_odds().pvalue:.2f}")
assert math.isclose(st.oddsratio_pooled, or_mh)
change = (or_wading - or_mh) / or_wading * 100
print(f"校正清理工作使 OR 從 {or_wading:.2f} 變為 {or_mh:.2f}（縮小 {change:.0f}%）")

**改變了什麼？為什麼？** 粗 OR（5.57）混合了兩件事：涉水本身的風險，以及清理工作額外的風險，因為清理工作者涉水涉得更多。在每一層裡，清理這個因素被固定住，而合併後的 MH OR 降到 **4.28**。涉水仍然與鉤端螺旋體病有強烈的關聯，關聯並沒有消失，但粗數字有一部分屬於清理工作。兩個分層的 OR（5.56 和 2.88）看起來不同，但以這些數字來說，檢定無法把它們和機遇區分開來（p ≈ 0.24），所以用一個 MH 彙整值是合理的。真實的研究有許多糾纏在一起的暴露和抽樣誤差，所以粗 OR 很少是全部的真相。

**Maya 向市衛生局長提出的建議：**

> 在 Amihan 之後涉過洪水，和鉤端螺旋體病有強烈的關聯，即使考慮了清理工作之後也一樣（校正後 OR 約為 4），而且在涉水的人當中，穿雨鞋與低很多的勝算有關。下一次洪水之前，市政府應該為必須涉水的清理人員與居民提供雨鞋和傷口照護，並告訴任何在接觸洪水後發燒的人及早就醫。在這項研究中，doxycycline 預防性投藥與較低的發病勝算有關，所以應該繼續依照 DOH 目前的指引，把它發給高暴露族群。

## 換你試試

1. **皮膚傷口。** `skin_wound` 的 OR 是否也被清理工作干擾了？*提示：複製 Mantel–Haenszel 的儲存格，把 `"waded_floodwater"` 改成 `"skin_wound"`。*
2. **男性真的風險比較高嗎？** `male` 的粗 OR 是 2.82。依 `waded_floodwater` 分層，並計算 MH OR。*提示：男性涉水的頻率比較高；預期 OR 會縮小。*
3. **故意重現 Tala 的錯誤。** 只把正確表格的列用 `table.iloc[::-1]` 翻轉，並計算 $ad/bc$。證明答案是 `1 / or_wading`。*提示：`math.isclose(...)` 是你的好幫手。*